# Overview of anomalies

In [1]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from anomaly.constants import GALAXY_LINES
from anomaly.utils import specobjid_to_idx
from anomaly.utils import VelocityFilter
from anomaly.utils import AnomalyOverlapAnalyzer
from autoencoders.ae import AutoEncoder

from sdss.metadata import MetaData

meta = MetaData()

# Constants

In [3]:
se_cols = [
    'mse',
    'mse_filter_250',
    # 'mse_filter_300',
    'mse_97',
    # 'mse_95',
    'mse_filter_250_97',
    # 'mse_filter_250_95',
    # 'mse_filter_300_97', 'mse_filter_300_95'
]

se_rank_cols = [
    f"rank_{col}" for col in se_cols
]

# ----------------------------------------------
rse_cols = [
    f"{col}_rel" for col in se_cols
]

rse_rank_cols = [
    f"rank_{col}_rel" for col in se_cols
]
# ----------------------------------------------
se_family = [
    'mse',
    'mse_97',
    # 'mse_95',
    'mse_filter_250',
    # 'mse_filter_300',
    'mse_filter_250_97',
    # 'mse_filter_250_95',
    # 'mse_filter_300_97', 'mse_filter_300_95'
]

rse_family = [
    f'{col}_rel' for col in se_family
]

# Custom functions

## IDs top anomalies

In [4]:
def get_ids(score, df, quantile=99, n_top=None, use_ntop=False):

    if use_ntop is False:
        
        quantile *= 0.01
        thresh = df[score].quantile(quantile)
        ids = set(df[df[score] > thresh].index)
        
    else:

        ids = set(
            df[score].sort_values(
                ascending=False
            ).iloc[:n_top].index
        )

    return ids

In [20]:
def top_unique_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}

    ids_top_list = []

    for score in scores_list:

        ids_set = get_ids(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

        ids_top_dict[score] = ids_set

        ids_top_list += list(ids_set)


    n_dictinct_top = len(set(ids_top_list))

    print(f"N unique: {n_dictinct_top}")


    unique_ids_dict = AnomalyOverlapAnalyzer.get_unique_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    for score in scores_list:
        n_unique = len(unique_ids_dict[score])

        unique_pct = n_unique/n_dictinct_top*100
        
        print(f"Unique to {score}:\n{n_unique} --> {unique_pct:.4f}%")

    return unique_ids_dict, ids_top_dict


In [21]:
def top_common_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}
    ids_top_list = []


    for score in scores_list:

        ids_set = get_ids(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

        ids_top_dict[score] = ids_set

        ids_top_list += list(ids_set)

    n_dictinct_top = len(set(ids_top_list))
    print(f"N unique: {n_dictinct_top}")

    common_ids_set = AnomalyOverlapAnalyzer.get_core_common_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    n_common = len(common_ids_set)
    common_pct = n_common/n_dictinct_top*100 
    print(f"N common:\n{n_common} -- > {common_pct:4f}%")

    return common_ids_set, ids_top_dict

## Figures

In [7]:
def anomaly_plot(wave, specs, objids, ranks, save_to):

    fig, ax = plt.subplots(
        figsize=(10, 5)
    )

    for spec, objid, rank in zip(specs, objids, ranks):

        print(f'Rank {rank:03d}', end='\r')

        ax.clear()

        ax.plot(wave, spec, color="black", label=f'Rank: {rank}')

        ax.minorticks_on()
        ax.set_xlabel(r"$\lambda$ [nm]")
        ax.set_title(f"Object ID: {objid}")

        ax.legend(
            loc='upper left',
            frameon=False,
        )

        fig.savefig(
            f"{save_to}/{rank:03d}_{objid}.jpeg",
            bbox_inches='tight'
        )

    plt.close(fig)

# Config

## Directories

In [8]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
models_dir = f"{data_dir}/models"
bin_id = 'bin_03'
#
ch_4_dir = f"{thesis_dir}/chapters/04_figures"
os.makedirs(f"{ch_4_dir}/{bin_id}", exist_ok=True)

## Data

In [9]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)

In [10]:
score_df = pd.read_csv(
    f"{scores_dir}/{bin_id}/scores_{bin_id}.csv.gz",
    index_col='specobjid'
)

rank = np.arange(score_df.shape[0]) + 1
score_rank_df = score_df.copy()
# score_rank_df
for col in score_df.columns:

    index_sorted = score_df.sort_values(
        by=col, ascending=False
    ).index

    score_rank_df.loc[index_sorted, f'rank_{col}'] = rank
    score_rank_df[f'rank_{col}'].astype(int)

n_spec = score_df.shape[0]
n_top_1_pct = int(n_spec*0.01)
n_top_1_pct, n_spec

(1818, 181850)

In [11]:
score = 'mse_97'
score_rank_df[[score, f'rank_{score}']].sort_values(
    by=score, ascending=False
).head(10)

,mse_97,rank_mse_97
specobjid,,
1919783100783552512,14.699505,1.0
2245159102839810048,13.236768,2.0
2930814289679771648,10.930245,3.0
1071977423131666432,10.445363,4.0
969534273977608192,9.671392,5.0
2008626123889469440,9.139043,6.0
1530151624956209152,8.159980,7.0
2811439757043722240,7.990755,8.0
1935588031146780672,7.488008,9.0


## Model

In [12]:
ae_model = AutoEncoder(
    reload=True,
    reload_from=f"{models_dir}/{bin_id}/winner",
)

In [13]:
ae_model.get_architecture_and_model_str()

['256_128_64_12_64_128_256', 'infoVae_rec_3776_alpha_0_lambda_9']

# Figs top anomalies

In [14]:
# # ```python
# n_top = 1000
# all_scores = se_cols + rse_cols
# plt.ioff()

# for score in all_scores:

#     specids_top_1 = score_df[score].sort_values(
#         ascending=False
#     ).index.to_numpy()[:n_top]

#     ranks = np.zeros(n_top).astype(int)

#     specs_top_1 = np.empty((n_top, wave.size))

#     for i, objid in enumerate(specids_top_1):

#         spec_idx = specobjid_to_idx(
#             objid, idx_id_spec
#         )

#         specs_top_1[i, :] = spectra[spec_idx, :]

#         ranks[i] = i

#     # -----------------------------------------------------------

#     save_to = f"{scores_dir}/{bin_id}/figs/{score}"

#     os.makedirs(save_to, exist_ok=True)

#     anomaly_plot(
#         wave_nm, specs=specs_top_1,
#         objids=specids_top_1, ranks=ranks,
#         save_to=save_to
#     )
# # ```

# No free lunch theorem

## IDs per score

In [15]:
ids_top_dict = {}

all_scores = se_cols + rse_cols

for score in all_scores:

    ids_top_dict[score] = get_ids(
        score=score,
        df=score_df.copy(),
        quantile=99,
        n_top=1000,
        use_ntop=False
    )

n_top_1 = len(ids_top_dict[score])
n_top_1

1819

# Distinct IDs

## SE family

In [22]:
se_unique_ids_dict, se_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=se_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N unique: 3173
Unique to mse:
433 --> 13.6464%
Unique to mse_filter_250:
261 --> 8.2257%
Unique to mse_97:
63 --> 1.9855%
Unique to mse_filter_250_97:
143 --> 4.5068%


In [23]:
score = 'mse'
unique_score_ids = list(se_unique_ids_dict[score])
score_rank_df.loc[
    unique_score_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False).head()

,mse,rank_mse
specobjid,,
2936445988153354240,16.863729,54.0
2032359622903359488,11.670377,134.0
624976740896237568,11.647491,135.0
305269539438880768,11.467894,143.0
1816077701509834752,11.392742,146.0


### Figs unique per SE

In [27]:
plt.ioff()

for score in se_cols:

    specids = list(se_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_se/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

## RSE family

In [24]:
rse_unique_ids_dict, chi_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=rse_cols,
    quantile=99,
    # n_top=1000, use_ntop=True
)

N unique: 3107
Unique to mse_rel:
309 --> 9.9453%
Unique to mse_filter_250_rel:
190 --> 6.1152%
Unique to mse_97_rel:
158 --> 5.0853%
Unique to mse_filter_250_97_rel:
177 --> 5.6968%


In [29]:
score = 'mse_97_rel'
unique_res_ids = list(rse_unique_ids_dict[score])
score_rank_df.loc[
    unique_res_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False).head()

,mse_97_rel,rank_mse_97_rel
specobjid,,
1781231151415846912,3.541895,666.0
511298502572140544,3.493030,766.0
2486033238892505088,3.486292,781.0
2553545448539318272,3.468160,815.0
2026696588415494144,3.455466,850.0


### Figs unique per RES

In [30]:
plt.ioff()

for score in rse_cols:

    specids = list(rse_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_rse/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

## All scores

In [25]:
all_scores = se_cols + rse_cols

all_unique_ids_dict, all_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=all_scores,
    quantile=99,
    # n_top=1000, use_ntop=True
)

N unique: 4095
Unique to mse:
262 --> 6.3980%
Unique to mse_filter_250:
128 --> 3.1258%
Unique to mse_97:
46 --> 1.1233%
Unique to mse_filter_250_97:
61 --> 1.4896%
Unique to mse_rel:
45 --> 1.0989%
Unique to mse_filter_250_rel:
125 --> 3.0525%
Unique to mse_97_rel:
64 --> 1.5629%
Unique to mse_filter_250_97_rel:
146 --> 3.5653%


In [34]:
score = 'mse'
unique_all_ids = list(all_unique_ids_dict[score])
score_rank_df.loc[
    unique_all_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False).head()

,mse,rank_mse
specobjid,,
2201231682403592192,9.990168,211.0
495462511089313792,9.045976,279.0
2291377342129399808,8.804863,299.0
2359927219364587520,8.752784,306.0
1934299677960726528,8.700253,309.0


### Figs unique among all

In [35]:
all_scores = se_cols + rse_cols
plt.ioff()

for score in all_scores:

    specids = list(all_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_all/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

# Common IDs

## SE family

In [26]:
se_common_ids, se_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=se_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N unique: 3173
N common:
692 -- > 21.809014%


In [51]:
scores = ['mse', 'mse_rel', 'mse_97', 'mse_97_rel']
rank_scores = [f'rank_{s}' for s in scores]
common_se_ids = list(se_common_ids)
score_rank_df.loc[
    common_se_ids, scores + rank_scores
].sort_values(by='mse', ascending=False).head()

,mse,mse_rel,mse_97,mse_97_rel,rank_mse,rank_mse_rel,rank_mse_97,rank_mse_97_rel
specobjid,,,,,,,,
1413149843194406912,46.845534,10.444906,5.173371,5.679945,2.0,49.0,52.0,31.0
1767703038748813312,45.294575,12.297180,5.532081,5.052953,4.0,31.0,35.0,63.0
2930814289679771648,44.142569,21.332923,10.930245,10.597543,5.0,7.0,3.0,4.0
1610143844655982592,42.411082,10.279599,4.401190,3.931915,6.0,54.0,143.0,305.0
305239577747023872,38.080637,24.499810,4.052517,4.232400,8.0,1.0,237.0,197.0


### Figs common among SE

In [43]:
specids = list(se_common_ids)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, f'rank_mse'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_ses/"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)

## RSE Family

In [28]:
rse_common_ids, res_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=rse_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N unique: 3107
N common:
789 -- > 25.394271%


In [49]:
scores = ['mse', 'mse_rel', 'mse_97', 'mse_97_rel']
rank_scores = [f'rank_{s}' for s in scores]

common_rse_ids = list(rse_common_ids)

score_rank_df.loc[
    common_rse_ids, scores + rank_scores
].sort_values(by='mse_rel', ascending=False).head()

,mse,mse_rel,mse_97,mse_97_rel,rank_mse,rank_mse_rel,rank_mse_97,rank_mse_97_rel
specobjid,,,,,,,,
305239577747023872,38.080637,24.499810,4.052517,4.232400,8.0,1.0,237.0,197.0
734111514142730240,31.560808,23.134280,5.029157,4.688281,14.0,4.0,63.0,118.0
3240467000396376064,20.224975,22.655893,4.501682,3.852538,39.0,5.0,121.0,349.0
2930814289679771648,44.142569,21.332923,10.930245,10.597543,5.0,7.0,3.0,4.0
1315165211342170112,29.042525,17.087980,4.432306,4.397494,20.0,10.0,138.0,163.0


### Figs common among RSE

In [52]:
specids = list(rse_common_ids)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, f'rank_mse_rel'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_rses/"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)

## All scores

In [27]:
all_scores = se_cols + rse_cols 
all_common_ids, all_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=all_scores,
    quantile=99,
    n_top=None, use_ntop=False
)

N unique: 4095
N common:
467 -- > 11.404151%


In [57]:
scores = ['mse', 'mse_rel', 'mse_97', 'mse_97_rel']
rank_scores = [f'rank_{s}' for s in scores]

common_all_ids = list(all_common_ids)
score_rank_df.loc[
    common_all_ids, scores + rank_scores
].sort_values(by='mse_rel', ascending=False).head()

,mse,mse_rel,mse_97,mse_97_rel,rank_mse,rank_mse_rel,rank_mse_97,rank_mse_97_rel
specobjid,,,,,,,,
305239577747023872,38.080637,24.499810,4.052517,4.232400,8.0,1.0,237.0,197.0
734111514142730240,31.560808,23.134280,5.029157,4.688281,14.0,4.0,63.0,118.0
3240467000396376064,20.224975,22.655893,4.501682,3.852538,39.0,5.0,121.0,349.0
2930814289679771648,44.142569,21.332923,10.930245,10.597543,5.0,7.0,3.0,4.0
1315165211342170112,29.042525,17.087980,4.432306,4.397494,20.0,10.0,138.0,163.0


### Figures

In [59]:
all_scores = se_cols + rse_cols
plt.ioff()


specids = list(all_common_ids)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, 'rank_mse'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_all"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)